In [9]:
import pandas as pd
import numpy as np

data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home',
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing',
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics',
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780,
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5,
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41,
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card',
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card',
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card',
                       'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)

print(df.info())
print("Missing values:\n", df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[ns]
 2   region            17 non-null     object        
 3   product_category  18 non-null     object        
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.4+ KB
None
Missing values:
 transaction_id      0
date                0
region              3
product_category    2
sales_amount        4
quantity            3
customer_age        4
payment_method      3
dtype: int64


In [10]:
df['region'] = df['region'].fillna(df['region'].mode()[0])
df['product_category'] = df['product_category'].fillna(df['product_category'].mode()[0])
df

,transaction_id,date,region,product_category,sales_amount,quantity,customer_age,payment_method
0,1,2024-10-01,North,Electronics,1200.0,2.0,25.0,Credit Card
1,2,2024-10-02,South,Clothing,450.0,5.0,34.0,UPI
2,3,2024-10-03,East,Clothing,890.0,3.0,NaN,Cash
3,4,2024-10-04,West,Books,NaN,1.0,45.0,Debit Card
4,5,2024-10-05,North,Electronics,1500.0,NaN,29.0,Credit Card
5,6,2024-10-06,North,Home,670.0,4.0,NaN,UPI
6,7,2024-10-07,South,Clothing,NaN,2.0,38.0,Cash
7,8,2024-10-08,North,Books,340.0,3.0,52.0,None
8,9,2024-10-09,East,Electronics,2100.0,1.0,27.0,Credit Card
9,10,2024-10-10,West,Clothing,780.0,5.0,41.0,UPI


In [11]:
df['sales_amount'] = df['sales_amount'].fillna(df['sales_amount'].median())
df['quantity'] = df['quantity'].fillna(method='ffill')
mean_age = round(df['customer_age'].mean())
df['customer_age'] = df['customer_age'].fillna(mean_age)
df = df.dropna(subset=['payment_method'])
print("Missing values after cleaning:\n", df.isna().sum())

Missing values after cleaning:
 transaction_id      0
date                0
region              0
product_category    0
sales_amount        0
quantity            0
customer_age        0
payment_method      0
dtype: int64


/tmp/ipykernel_307/1213428598.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['quantity'] = df['quantity'].fillna(method='ffill')


In [12]:
total_sales_region = df.groupby('region')['sales_amount'].sum()
print("Total Sales by Region:\n", total_sales_region)


Total Sales by Region:
 region
East     3790.0
North    6460.0
South    1900.0
West     2230.0
Name: sales_amount, dtype: float64


In [13]:
avg_sales_category = df.groupby('product_category')['sales_amount'].mean()
print("\nAverage Sales by Category:\n", avg_sales_category)


Average Sales by Category:
 product_category
Books           508.333333
Clothing        680.000000
Electronics    1381.250000
Home            812.500000
Name: sales_amount, dtype: float64


In [14]:
grouped = df.groupby(['region', 'product_category']).agg({
    'sales_amount': 'sum',
    'quantity': 'sum'
}).reset_index()

print("Region-Category Summary:\n", grouped)

Region-Category Summary:
   region product_category  sales_amount  quantity
0   East            Books         800.0       4.0
1   East         Clothing         890.0       3.0
2   East      Electronics        2100.0       1.0
3  North         Clothing         510.0       3.0
4  North      Electronics        2700.0       3.0
5  North             Home        3250.0      12.0
6  South         Clothing        1900.0       9.0
7   West            Books         725.0       1.0
8   West         Clothing         780.0       5.0
9   West      Electronics         725.0       1.0


In [15]:
top3 = grouped.sort_values(by='sales_amount', ascending=False).head(3)
print("\nTop 3 Combinations:\n", top3)


Top 3 Combinations:
   region product_category  sales_amount  quantity
5  North             Home        3250.0      12.0
4  North      Electronics        2700.0       3.0
2   East      Electronics        2100.0       1.0


In [16]:
def sales_range(series):
    return series.max() - series.min()

In [17]:
range_by_region = df.groupby('region')['sales_amount'].apply(sales_range)
print("Sales Range by Region:", range_by_region)

Sales Range by Region: region
East     1720.0
North     990.0
South     275.0
West       55.0
Name: sales_amount, dtype: float64


In [21]:
region_summary = df.groupby('region').agg({
    'sales_amount': ['sum', 'mean', 'max'],
    'quantity': ['sum', 'min']
})

print("\nRegion Summary Metrics:\n", region_summary)


Region Summary Metrics:
        sales_amount                     quantity     
                sum        mean     max      sum  min
region                                               
East         3790.0  947.500000  2100.0      8.0  1.0
North        6460.0  922.857143  1500.0     18.0  1.0
South        1900.0  633.333333   725.0      9.0  2.0
West         2230.0  743.333333   780.0      7.0  1.0
